# 중심 입력 비교 학습

기존 모델과 같은 자료·설정으로 중앙 64/32 픽셀만 읽어 비교합니다. 원본과 라벨은 바꾸지 않습니다. 작은 입력은 주변 결함을 줄이지만 BPD의 선 모양 문맥을 잃을 수도 있습니다.

이미 출력이 보이면 비교가 완료된 상태입니다. 다시 전체 실행하면 새로운 비교 학습이 시작됩니다. 기존 노트북은 보존됩니다. SiC XRT GPU 커널에서 실행하세요.


In [ ]:
import os
from pathlib import Path

from IPython.display import Image, display

from sic_xrt_ml.training.center_comparison import compare_centers

DATASET_ROOT = Path(os.environ.get("SIC_XRT_DATASET_ROOT", "CHANGE_ME"))
BASELINE_RUN = Path(os.environ.get("SIC_XRT_BASELINE_RUN", "CHANGE_ME"))
REPO = Path(os.environ["SIC_XRT_REPO"])
assert DATASET_ROOT.is_dir() and BASELINE_RUN.is_dir(), "데이터셋과 기존 학습 결과 폴더를 지정하세요."


## 비교 실행

기존 128 입력 결과를 보존하고 64/32 입력을 각각 같은 횟수로 학습합니다. 시험셋은 평가하지 않습니다. 화면의 반복 횟수와 진행 상황을 확인하세요.


In [ ]:
comparison = compare_centers(DATASET_ROOT, BASELINE_RUN, REPO / "runs", crops=(64, 32), device="cuda")


## 결과 확인

검증 macro F1과 클래스별 F1을 함께 봅니다. 이 검증 웨이퍼는 이미 실험 선택에 사용됐으므로 점수는 탐색 결과입니다. 새 웨이퍼 전체 성능을 입증하지 않습니다.


In [ ]:
for record in comparison["records"]:
    print(f"입력 {record['input_size']} | 최선 회차 {record['selected_epoch']} | 검증 macro F1 {record['metrics']['macro_f1']:.3f}")
    print({name: round(value['f1'], 3) for name, value in record['metrics']['per_class'].items()})
print("결과 폴더:", comparison["output_dir"])
print("시험셋 평가:", comparison["test_evaluated"])
display(Image(filename=str(Path(comparison["output_dir"]) / "comparison.png")))
